# Exploratory Notebook — Negation Failures in LLMs

Quick experiments: load a model, run prompts, inspect top predictions.

In [ ]:
import sys
import os

# Ensure project root is on the path
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np

# Reproducibility
torch.manual_seed(68)
np.random.seed(68)

## 1. Load Model

In [ ]:
from src.models.load_models import load_model

model = load_model("gpt2-small")

## 2. Run a Prompt and Print Top Predictions

In [ ]:
prompt = "The capital of France is"

tokens = model.to_tokens(prompt)
logits = model(tokens)
last_logits = logits[0, -1, :]

# Get top 10 predictions
top_k = torch.topk(last_logits, k=10)
print(f"Prompt: {prompt!r}\n")
print("Top 10 predictions:")
for i, (logit, idx) in enumerate(zip(top_k.values, top_k.indices)):
    token_str = model.to_string(idx.item())
    print(f"  {i+1:2d}. {token_str!r:>15s}  (logit: {logit.item():.4f})")

## 3. Compare Positive vs. Negated Prompt

In [ ]:
from src.dataset.build_prompts import build_positive_prompt, build_negated_prompt
from src.benchmark.metrics import get_top_prediction, extract_target_logit

positive = build_positive_prompt("The capital of France is")
negated = build_negated_prompt("The capital of France is")
target = "Paris"

print(f"Positive prompt: {positive!r}")
print(f"Negated prompt:  {negated!r}")
print(f"Target:          {target!r}")
print()

with torch.no_grad():
    pos_pred = get_top_prediction(model, positive)
    neg_pred = get_top_prediction(model, negated)
    pos_logit = extract_target_logit(model, positive, target)
    neg_logit = extract_target_logit(model, negated, target)

print(f"Positive prediction: {pos_pred!r}  (target logit: {pos_logit:.4f})")
print(f"Negated prediction:  {neg_pred!r}  (target logit: {neg_logit:.4f})")

## 4. Load Dataset Sample

In [ ]:
from src.dataset.load_dataset import load_counterfact

data = load_counterfact(dataset_size=5)
for i, entry in enumerate(data):
    print(f"{i+1}. prompt: {entry['prompt']!r}")
    print(f"   target: {entry['target']!r}")
    print()